# 레슨 02 — 실습 미션 정답지

> 🔒 **교사·관리자 전용. 학생에게 배포 금지.**

---

## 환경 셀

In [ ]:
import os
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
DATA_BASE = "./data"
np.random.seed(42)
print("data base:", DATA_BASE)

---

## 미션 1 정답 — 2D 배열 구조 파악

In [ ]:
raw = np.loadtxt(f"{DATA_BASE}/student_scores_2d.csv", delimiter=",", skiprows=1)
print("처음 5행:\n", raw[:5])

# 2. 구조 확인
print("shape:", raw.shape)    # (120, 5)
print("ndim:",  raw.ndim)     # 2
print("size:",  raw.size)     # 600

# 3. 점수 열만 추출 (열 2, 3, 4)
scores = raw[:, 2:]
print("scores.shape:", scores.shape)   # (120, 3)

# 4. 1D 로 변환
flat = scores.reshape(-1)
print("flat 길이:", len(flat))          # 360

# 5. 전치
print("scores.T.shape:", scores.T.shape)  # (3, 120)

**채점 포인트**

- `raw[:, 2:]` 로 열을 올바르게 슬라이싱했는가.
- `reshape(-1)` 과 `.T` 의 개념을 구분했는가.

**흔한 실수**: `raw[:, 2:5]` 는 정답이지만 `raw[:, 2:4]` 는 science 열이 빠진다.

---

## 미션 2 정답 — 축 방향 집계와 표준화

In [ ]:
class_id = raw[:, 0].astype(int)

# 1. 과목별 평균·표준편차
col_mean = scores.mean(axis=0)
col_std  = scores.std(axis=0)
for i, name in enumerate(["math","english","science"]):
    print(f"{name}: 평균 {col_mean[i]:.1f}  표준편차 {col_std[i]:.1f}")

# 2. 반별 수학 점수 평균
math_scores = scores[:, 0]
for c in [1, 2, 3]:
    mask = (class_id == c)
    print(f"{c}반 수학 평균: {math_scores[mask].mean():.1f}")

# 3. z-점수 표준화
z = (scores - col_mean) / col_std
print("표준화 후 평균:", z.mean(axis=0).round(4))   # ≈ [0, 0, 0]
print("표준화 후 표준편차:", z.std(axis=0).round(4)) # ≈ [1, 1, 1]

# 4. 수학 평균이 가장 높은 반
class_means = np.array([scores[class_id==c, 0].mean() for c in [1,2,3]])
print("반별 수학 평균:", class_means.round(1))
print("최고 반:", class_means.argmax() + 1, "반")   # 기대: 3반(base 76)

**채점 포인트**

- `axis=0` (과목별) 과 `axis=1` (학생별)을 정확히 구분했는가.
- `(scores - col_mean) / col_std` 브로드캐스팅이 동작하는 이유를 설명할 수 있는가.

---

## 미션 3 정답 — 브로드캐스팅 탐구

In [ ]:
a = np.array([[1,2,3],[4,5,6],[7,8,9]])
b = np.array([10, 20, 30])

# 2. a + b
print(a + b)
# [[11, 22, 33],
#  [14, 25, 36],
#  [17, 28, 39]]

# 3. a * b 후 각 행 합계
print((a * b).sum(axis=1))
# [10*1+20*2+30*3, 10*4+20*5+30*6, 10*7+20*8+30*9]
# [140, 320, 500]

# 4. 최솟값 정규화
a_min = a.min(axis=0)           # [1, 2, 3]
a_normalized = a - a_min
print("최솟값 정규화:\n", a_normalized)
# [[0, 0, 0],
#  [3, 3, 3],
#  [6, 6, 6]]

# 5. 열벡터 c
c = np.array([[100],[200],[300]])   # shape (3, 1)
print("a + c:\n", a + c)
# c의 shape: (3,1)  a의 shape: (3,3)
# → axis=1 방향으로 c가 늘어나 각 행에 100, 200, 300 이 더해짐
# [[101, 102, 103],
#  [204, 205, 206],
#  [307, 308, 309]]

**채점 포인트**

- (3,) 과 (3,1) 의 차이를 설명할 수 있는가. (열벡터 vs 1D 벡터)
- `min(axis=0)` 방향을 틀리지 않았는가.

---

## 미션 4 정답 — 주사위 시뮬레이션

In [ ]:
np.random.seed(2024)
N = 10000
die1 = np.random.randint(1, 7, N)
die2 = np.random.randint(1, 7, N)
total = die1 + die2

# 2. 합 7 비율
p7 = (total == 7).mean()
print(f"합 7 비율: {p7:.4f}  (이론: {1/6:.4f}  오차: {abs(p7 - 1/6):.4f})")

# 3. n별 수렴
for n in [10, 100, 1000, 10000]:
    np.random.seed(2024)
    d1 = np.random.randint(1, 7, n)
    d2 = np.random.randint(1, 7, n)
    t  = d1 + d2
    print(f"n={n:>6}  합 평균 {t.mean():.4f}  (기댓값 7.0000)")

# 4. CSV 빈도
raw_dice = np.loadtxt(f"{DATA_BASE}/dice_rolls.csv", delimiter=",", skiprows=1)
totals   = raw_dice[:, 3].astype(int)
for s in range(2, 13):
    cnt = (totals == s).sum()
    bar = "█" * round(cnt / len(totals) * 100)
    print(f"합 {s:2d}: {cnt:5d}회 {bar}")

**예상 출력(합 7)**
```
합 7: 1668회 (가장 많음)
```

**채점 포인트**

- seed 를 루프 안에서 재설정해야 매 n 에서 동일한 시작점인지 여부를 토론한다 (교사 재량).
- "이론 확률 6/36 = 1/6" 의 근거를 학생이 설명할 수 있는지 확인한다.

---

## 미션 5 정답 — 주가 랜덤워크 분석

In [ ]:
raw_stock = np.loadtxt(f"{DATA_BASE}/stock_prices.csv", delimiter=",", skiprows=1)
days   = raw_stock[:, 0].astype(int)
prices = raw_stock[:, 1:]
names  = ["TechA","FinB","SmallC","MidD","LargeE"]

# 1. 시작·최종·최고·최저
print(f"{'종목':<10} {'시작':>8} {'최종':>8} {'최고':>8} {'최저':>8}")
for i, n in enumerate(names):
    print(f"{n:<10} {int(prices[0,i]):>8,} {int(prices[-1,i]):>8,}"
          f" {int(prices[:,i].max()):>8,} {int(prices[:,i].min()):>8,}")

# 2. 일간 로그수익률
log_ret = np.diff(np.log(prices), axis=0)   # shape (251, 5)

# 3. 평균·표준편차
ret_mean = log_ret.mean(axis=0)
ret_std  = log_ret.std(axis=0)
print("\n일간 수익률:")
for i, n in enumerate(names):
    print(f"  {n}: 평균 {ret_mean[i]*100:+.3f}%  표준편차 {ret_std[i]*100:.3f}%")

# 4. 연환산
ann_ret = ret_mean * 252
ann_vol = ret_std  * np.sqrt(252)
print("\n연환산:")
for i, n in enumerate(names):
    print(f"  {n}: 수익률 {ann_ret[i]*100:+.1f}%  변동성 {ann_vol[i]*100:.1f}%")

# 5. argmax
print(f"\n변동성 최대: {names[ann_vol.argmax()]}")
print(f"수익률 최고: {names[ann_ret.argmax()]}")

**결론 예시**

In [ ]:
%%markdown
## 결론

1. **변동성 분석**: SmallC 가 연환산 변동성이 가장 높아 고위험·고수익 성격의 종목이다.
   투자 성향이 공격적인 고객에게는 SmallC, 안정을 원한다면 LargeE 를 권장한다.

---

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 코멘트 |
|---|---|---|
| `scores[:][2:]` 로 시도 | axis 개념 혼동 | `scores[:, 2:]` 가 맞다고 시연 |
| `axis` 없이 `mean()` 사용 | 전체 평균 계산 | axis 방향 재설명 |
| reshape 에 맞지 않는 크기 | 원소 수 미확인 | size 먼저 확인하는 습관 |
| `std` 분모 ddof 혼동 | 표본/모집단 | 레슨 01 ddof 박스 참조 |
| 브로드캐스팅 (3,) vs (3,1) | 벡터 방향 혼동 | `reshape(-1,1)` 로 열벡터 변환 시연 |
| cumsum axis 누락 | 1D 로 flatten 됨 | `axis=0` 강조 |

---

## 보너스 미션 정답 가이드

보너스는 필수 채점 대상은 아니지만 빠른 학생에게 좋은 확장 과제다. 아래 코드는 교사용 확인용이다.

### B1 — 센서 그리드

In [ ]:
raw_sensor = np.loadtxt(f"{DATA_BASE}/sensor_grid.csv", delimiter=",", skiprows=1)
temps = raw_sensor[:, 2].reshape(20, 30)

hot_col_by_row = temps.argmax(axis=1)
row_max_temp = temps.max(axis=1)
for r in range(20):
    print(f"row {r:02d}: hottest col={hot_col_by_row[r]:02d}, temp={row_max_temp[r]:.1f}℃")

**채점 포인트**: `reshape(20, 30)` 전 원소 수가 600 인지 확인했는가. `axis=1` 은 각 행에서 열 방향으로 최댓값을 찾는다는 뜻이다.

### B2 — 복수 경로 랜덤워크

In [ ]:
np.random.seed(77)
steps = np.random.choice([-1, 1], size=(252, 100))
walks = np.cumsum(steps, axis=0)
final_pos = walks[-1]

print("최종 위치 평균:", final_pos.mean())
print("최종 위치 표준편차:", final_pos.std())
print("이론 표준편차:", np.sqrt(252))

**해석 포인트**: 평균은 0 근처, 표준편차는 `sqrt(252)` 근처에 오면 정상이다. 시행 수가 100개라 완전히 일치하지는 않는다.

### B3 — 최대 연속 상승

In [ ]:
tech = prices[:, 0]
up = np.diff(tech) > 0
changes = np.diff(np.r_[False, up, False].astype(int))
starts = np.where(changes == 1)[0]
ends = np.where(changes == -1)[0] - 1
lengths = ends - starts + 1

if lengths.size == 0:
    print("연속 상승 구간 없음")
else:
    best = lengths.argmax()
    print("최대 연속 상승일:", int(lengths[best]))
    print("시작 인덱스:", int(starts[best]), "끝 인덱스:", int(ends[best]))

이 문제는 1강 보너스에서 다룬 boolean 배열의 시작/끝 탐색을 2강 주가 데이터에 적용하는 문제다. 정답 수치보다 `np.r_` 로 양끝을 패딩하고 `diff` 로 구간 경계를 찾는 흐름을 설명할 수 있는지 본다.